# Cricket Match Results Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

# load the 3 csv files
bat = pd.read_csv('MatchResults_Batting.csv')
tbat = pd.read_csv('MatchResults_TeamBat.csv')
bowl = pd.read_csv('MatchResults_Bowling.csv')

print("Batting:", bat.shape)
print("TeamBat:", tbat.shape)
print("Bowling:", bowl.shape)

In [ ]:
# clean spaces and fix team name spelling differences
team_fix = {
    'Afganisthan': 'Afghanistan',
    'Afganistan': 'Afghanistan',
    'ENG': 'England',
    'New zealand': 'New Zealand',
    'Zimbabwae': 'Zimbabwe',
    'Zimbawe': 'Zimbabwe',
    'Netherland': 'Netherlands'
}

for df in [bat, tbat, bowl]:
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip().replace('nan', np.nan)
    for col in ['Team', 'Vs']:
        if col in df.columns:
            df[col] = df[col].replace(team_fix)

# clean player names (remove (c) and † symbols)
bat['Player'] = bat['Player'].str.replace('(c)', '', regex=False).str.replace('†', '', regex=False).str.strip()
bowl['Bowler'] = bowl['Bowler'].dropna().str.replace('(c)', '', regex=False).str.replace('†', '', regex=False).str.strip()

# convert numeric columns properly
for col in ['R', 'B', '4s', '6s', 'SR', 'Chase', 'Win']:
    bat[col] = pd.to_numeric(bat[col], errors='coerce')

for col in ['R', 'B', '4s', '6s', 'SR', 'Chase', 'Total', 'Win']:
    tbat[col] = pd.to_numeric(tbat[col], errors='coerce')

for col in ['O', 'M', 'R', 'W', 'Econ', 'Dots', '4s', '6s', 'Wide', 'NB', 'Win']:
    bowl[col] = pd.to_numeric(bowl[col], errors='coerce')

# remove summary rows in Bowling where Bowler is empty
bowl = bowl[bowl['Bowler'].notna() & bowl['O'].notna()].copy()

print("Data cleaned successfully!")

### 1. Top 10 batsmen based on total runs scored

In [ ]:
q1 = bat.groupby(['Player', 'Team'], as_index=False).agg(
    Innings=('R', 'count'),
    Total_Runs=('R', 'sum'),
    Balls=('B', 'sum'),
    Fours=('4s', 'sum'),
    Sixes=('6s', 'sum')
)
q1['SR'] = ((q1['Total_Runs'] / q1['Balls']) * 100).round(2)
q1 = q1.sort_values('Total_Runs', ascending=False).head(10)

print(q1.to_string(index=False))

plt.figure(figsize=(9, 4))
sns.barplot(data=q1, x='Total_Runs', y='Player')
plt.title('Top 10 Batsmen by Total Runs')
plt.xlabel('Total Runs')
plt.ylabel('Player')
plt.show()

### 2. Top 10 players with highest strike rate (minimum 50 balls faced)

In [ ]:
players = bat.groupby(['Player', 'Team'], as_index=False).agg(
    Innings=('R', 'count'),
    Total_Runs=('R', 'sum'),
    Balls=('B', 'sum')
)

q2 = players[players['Balls'] >= 50].copy()
q2['Strike_Rate'] = ((q2['Total_Runs'] / q2['Balls']) * 100).round(2)
q2 = q2.sort_values('Strike_Rate', ascending=False).head(10)

print(q2.to_string(index=False))

plt.figure(figsize=(9, 4))
sns.barplot(data=q2, x='Strike_Rate', y='Player')
plt.title('Top 10 Strike Rates (Min 50 Balls)')
plt.xlabel('Strike Rate')
plt.ylabel('Player')
plt.show()

### 3. Player with most fours and player with most sixes

In [ ]:
bounds = bat.groupby(['Player', 'Team'], as_index=False)[['4s', '6s', 'R']].sum()

print("Top 5 Most Fours:")
print(bounds.sort_values('4s', ascending=False).head(5).to_string(index=False))

print("\nTop 5 Most Sixes:")
print(bounds.sort_values('6s', ascending=False).head(5).to_string(index=False))

### 4. Team with highest number of match wins

In [ ]:
# get unique team-match results
matches = bat[['Date', 'Team', 'Vs', 'Chase', 'Win']].drop_duplicates()

q4 = matches.groupby('Team', as_index=False).agg(
    Matches=('Win', 'count'),
    Wins=('Win', 'sum')
)
q4['Win_Pct'] = ((q4['Wins'] / q4['Matches']) * 100).round(2)
q4 = q4.sort_values(['Wins', 'Win_Pct'], ascending=False)

print(q4.to_string(index=False))

plt.figure(figsize=(9, 5))
sns.barplot(data=q4, x='Wins', y='Team')
plt.title('Match Wins by Team')
plt.xlabel('Wins')
plt.ylabel('Team')
plt.show()

### 5. Highest run scorer for each match

In [ ]:
# standardize date string like 7.2.26 vs 07.02.26
bat['Date_Clean'] = bat['Date'].apply(lambda d: '.'.join([str(int(x)) for x in str(d).split('.')]))

# create a unique match id for each match
bat['Match_ID'] = bat.apply(lambda r: r['Date_Clean'] + ' | ' + ' vs '.join(sorted([r['Team'], r['Vs']])), axis=1)

# find max runs in each match
max_runs = bat.groupby('Match_ID')['R'].transform('max')
q5 = bat[bat['R'] == max_runs][['Date', 'Player', 'Team', 'Vs', 'R']].drop_duplicates()

print("Total matches:", len(q5))
print(q5.to_string(index=False))

### 6. Chasing vs Batting First performance comparison

In [ ]:
# individual batting comparison (0 = Bat First, 1 = Chase)
ind_comp = bat.groupby('Chase').agg(
    Innings=('R', 'count'),
    Avg_Player_Runs=('R', 'mean'),
    Avg_SR=('SR', 'mean')
).round(2)
print("Individual Batting (0 = Bat First, 1 = Chase):")
print(ind_comp)

# team totals from TeamBat summary rows (where Player is NaN)
team_totals = tbat[tbat['Player'].isna() & tbat['Chase'].isin([0.0, 1.0])].copy()
team_comp = team_totals.groupby('Chase').agg(
    Team_Innings=('Total', 'count'),
    Avg_Team_Runs=('Total', 'mean'),
    Median_Team_Runs=('Total', 'median'),
    Wins=('Win', 'sum'),
    Win_Pct=('Win', lambda x: x.mean() * 100)
).round(2)

print("\nTeam Totals Comparison (0 = Bat First, 1 = Chase):")
print(team_comp)

### 7. Top 10 bowlers by total wickets and average economy rate

In [ ]:
# convert cricket overs (like 3.5) into total balls bowled
def get_balls(overs):
    w = int(overs)
    b = round((overs - w) * 10)
    return w * 6 + b

bowl['Balls'] = bowl['O'].apply(get_balls)

q7 = bowl.groupby(['Bowler', 'Team'], as_index=False).agg(
    Matches=('W', 'count'),
    Balls=('Balls', 'sum'),
    Runs_Conceded=('R', 'sum'),
    Wickets=('W', 'sum'),
    Avg_Match_Econ=('Econ', 'mean')
)

q7['True_Econ'] = ((q7['Runs_Conceded'] / (q7['Balls'] / 6))).round(2)
q7['Avg_Match_Econ'] = q7['Avg_Match_Econ'].round(2)
q7 = q7.sort_values(['Wickets', 'True_Econ'], ascending=[False, True]).head(10)

print(q7[['Bowler', 'Team', 'Matches', 'Wickets', 'True_Econ', 'Avg_Match_Econ']].to_string(index=False))

plt.figure(figsize=(9, 4))
sns.barplot(data=q7, x='Wickets', y='Bowler')
plt.title('Top 10 Bowlers by Total Wickets')
plt.xlabel('Wickets')
plt.ylabel('Bowler')
plt.show()

### 8. Most runs through boundaries and strike rate comparison

In [ ]:
bat_stats = bat.groupby(['Player', 'Team'], as_index=False).agg(
    Runs=('R', 'sum'),
    Balls=('B', 'sum'),
    Fours=('4s', 'sum'),
    Sixes=('6s', 'sum')
)
bat_stats['SR'] = ((bat_stats['Runs'] / bat_stats['Balls']) * 100).round(2)
bat_stats['Boundary_Runs'] = bat_stats['Fours'] * 4 + bat_stats['Sixes'] * 6
bat_stats['Boundary_Pct'] = np.where(bat_stats['Runs'] > 0, (bat_stats['Boundary_Runs'] / bat_stats['Runs']) * 100, 0).round(2)

top10_bound = bat_stats.sort_values('Boundary_Runs', ascending=False).head(10)
others = bat_stats[(~bat_stats['Player'].isin(top10_bound['Player'])) & (bat_stats['Balls'] >= 20)]

print(top10_bound[['Player', 'Team', 'Runs', 'Fours', 'Sixes', 'Boundary_Runs', 'Boundary_Pct', 'SR']].to_string(index=False))
print("\nTop 10 Boundary Scorers Avg SR:", round(top10_bound['SR'].mean(), 2))
print("Other Batsmen (>=20 balls) Avg SR:", round(others['SR'].mean(), 2))

### 9. Top 20 batsmen using Player Performance Score

In [ ]:
q9 = bat_stats[bat_stats['Balls'] >= 30].copy()

scaler = MinMaxScaler()
q9[['n_runs', 'n_sr', 'n_4s', 'n_6s', 'n_balls']] = scaler.fit_transform(q9[['Runs', 'SR', 'Fours', 'Sixes', 'Balls']])

# weighted score: 35% runs, 25% SR, 15% fours, 15% sixes, 10% balls faced
q9['Performance_Score'] = (
    0.35 * q9['n_runs'] +
    0.25 * q9['n_sr'] +
    0.15 * q9['n_4s'] +
    0.15 * q9['n_6s'] +
    0.10 * q9['n_balls']
) * 100

top20_bat = q9.sort_values('Performance_Score', ascending=False).head(20)
print(top20_bat[['Player', 'Team', 'Runs', 'Balls', 'SR', 'Fours', 'Sixes', 'Performance_Score']].round(2).to_string(index=False))

### 10. Top 10 all-round performers using Match Impact Score

In [ ]:
bowl_stats = bowl.groupby(['Bowler', 'Team'], as_index=False).agg(
    Balls_Bowled=('Balls', 'sum'),
    Runs_Conceded=('R', 'sum'),
    Wickets=('W', 'sum')
)
bowl_stats['Econ'] = ((bowl_stats['Runs_Conceded'] / (bowl_stats['Balls_Bowled'] / 6))).round(2)

# combine batting and bowling tables
allround = pd.merge(bat_stats, bowl_stats, left_on=['Player', 'Team'], right_on=['Bowler', 'Team'])

# keep players who batted at least 20 balls and bowled at least 2 overs (12 balls)
allround = allround[(allround['Balls'] >= 20) & (allround['Balls_Bowled'] >= 12)].copy()

# calculate win rate per player
p_matches = pd.concat([
    bat[['Date', 'Team', 'Player', 'Win']],
    bowl[['Date', 'Team', 'Bowler', 'Win']].rename(columns={'Bowler': 'Player'})
]).drop_duplicates(subset=['Date', 'Team', 'Player'])

wins = p_matches.groupby(['Player', 'Team'], as_index=False).agg(
    Matches=('Win', 'count'),
    Wins=('Win', 'sum')
)
wins['Win_Rate'] = wins['Wins'] / wins['Matches']

allround = pd.merge(allround, wins, on=['Player', 'Team'])

scaler = MinMaxScaler()
allround['n_runs'] = scaler.fit_transform(allround[['Runs']])
allround['n_wkts'] = scaler.fit_transform(allround[['Wickets']])
allround['n_sr'] = scaler.fit_transform(allround[['SR']])
allround['n_econ'] = 1 - scaler.fit_transform(allround[['Econ']])
allround['n_bound'] = scaler.fit_transform(allround[['Boundary_Runs']])
allround['n_win'] = scaler.fit_transform(allround[['Win_Rate']])

# Match Impact Score: 25% runs, 25% wickets, 15% SR, 15% economy, 10% boundary runs, 10% win rate
allround['Impact_Score'] = (
    0.25 * allround['n_runs'] +
    0.25 * allround['n_wkts'] +
    0.15 * allround['n_sr'] +
    0.15 * allround['n_econ'] +
    0.10 * allround['n_bound'] +
    0.10 * allround['n_win']
) * 100

top10_allround = allround.sort_values('Impact_Score', ascending=False).head(10)
print(top10_allround[['Player', 'Team', 'Matches', 'Runs', 'SR', 'Boundary_Runs', 'Wickets', 'Econ', 'Wins', 'Impact_Score']].round(2).to_string(index=False))

plt.figure(figsize=(9, 4))
sns.barplot(data=top10_allround, x='Impact_Score', y='Player')
plt.title('Top 10 All-Rounders by Match Impact Score')
plt.xlabel('Match Impact Score (0 - 100)')
plt.ylabel('Player')
plt.show()